# Minimal Segmentation
- This allows you to segment vasculature and explore the t0, t2 signal intensities and the segmented vasculature
- You can take images and make movies of key frames
- Run this on the nap-ij-record kernel

In [1]:
import matplotlib.pyplot as plt
import numpy as np
import math
import os
import pandas as pd
from scipy import ndimage as ndi
from skimage.transform import rescale
from skimage.measure import marching_cubes, mesh_surface_area, label, regionprops, regionprops_table
from skimage import util, measure
import napari
from tifffile import imread
from liffile import LifFile
from pathlib import Path
import imagej
from imagej import Mode
import scyjava as sj
from matplotlib.colors import to_rgba
from tkinter import Tk, filedialog
from pathlib import Path
import pandas as pd
# import warnings
# warnings.filterwarnings("ignore")

ij = imagej.init("sc.fiji:fiji", mode=Mode.INTERACTIVE, add_legacy=True)

IJ = sj.jimport("ij.IJ")
Duplicator = sj.jimport("ij.plugin.Duplicator")
WekaSegmentation = sj.jimport("trainableSegmentation.WekaSegmentation")
ImagePlus = sj.jimport("ij.ImagePlus")
ImageStack = sj.jimport("ij.ImageStack")
FloatProcessor = sj.jimport("ij.process.FloatProcessor")

In [2]:
def numpy_to_imageplus(arr: np.ndarray, title="image") -> ImagePlus:
    if arr.ndim == 2:
        y, x = arr.shape
        pix = np.asarray(arr, dtype=np.float32).ravel()
        java_floats = sj.jarray("f", pix.size)
        # fill java float[]
        for i, v in enumerate(pix.tolist()):
            java_floats[i] = float(v)
        fp = FloatProcessor(x, y, java_floats)
        return ImagePlus(title, fp)

    if arr.ndim == 3:
        z, y, x = arr.shape
        stack = ImageStack(x, y)
        for zi in range(z):
            pix = np.asarray(arr[zi], dtype=np.float32).ravel()
            java_floats = sj.jarray("f", pix.size)
            for i, v in enumerate(pix.tolist()):
                java_floats[i] = float(v)
            fp = FloatProcessor(x, y, java_floats)
            stack.addSlice(fp)
        return ImagePlus(title, stack)

    raise ValueError(f"Unsupported shape: {arr.shape}")

In [3]:
def imageplus_to_numpy(imp: ImagePlus) -> np.ndarray:
    w, h = imp.getWidth(), imp.getHeight()
    n = imp.getStackSize()
    stack = imp.getStack()
    out = []
    for z in range(1, n + 1):
        ip = stack.getProcessor(z)
        pix = np.array(ip.getPixels()).reshape(h, w)
        out.append(pix)
    return np.stack(out, axis=0) if n > 1 else out[0]

In [4]:
def convert(img, target_type_min, target_type_max, target_type):
    """
    Converts an image to a specified data type while scaling its intensity values.

    This function rescales the intensity values of an image from its original range 
    to a new target range specified by `target_type_min` and `target_type_max`, and 
    then converts it to the desired data type.

    This step is required as deconvolved images are not always scaled 0->255! 

    Parameters:
    -----------
    img : numpy.ndarray
        The input image array to be converted.
    target_type_min : int or float
        The minimum value of the target intensity range.
    target_type_max : int or float
        The maximum value of the target intensity range.
    target_type : numpy.dtype
        The desired data type of the output image (e.g., np.uint8, np.float32).

    Returns:
    --------
    new_img : numpy.ndarray
        The rescaled image with values mapped to the new intensity range and converted 
        to the specified data type.

    Notes:
    ------
    - This function performs a linear transformation to scale pixel values.
    - It ensures that the output values are properly mapped between `target_type_min` and 
      `target_type_max`.
    """
    imin = img.min()
    imax = img.max()

    a = (target_type_max - target_type_min) / (imax - imin)
    b = target_type_max - a * imax
    new_img = (a * img + b).astype(target_type)
    return new_img

In [5]:
def apply_weka_with_exact_preprocessing(image: np.ndarray, model_path: str) -> np.ndarray:
    imp = numpy_to_imageplus(image, title="image")
    dup = Duplicator().run(imp)
    IJ.run(dup, "8-bit", "")
    IJ.run(dup, "Auto Threshold", "method=Otsu stack")
    IJ.run(dup, "Erode (3D)", "iso=255")

    try:
        seg = WekaSegmentation(dup)
        seg.loadClassifier(model_path)
        out_imp = seg.applyClassifier(dup, 0, False)
        if out_imp is None:
            out_imp = seg.getClassifiedImage()

        if out_imp is None:
            raise RuntimeError("No classified image returned (out_imp is None). Likely a Java-side error or model/input mismatch.")

        return imageplus_to_numpy(out_imp)

    except:
        # Show Java exception details if present
        print("=== Python/Java exception ===")

In [6]:
def step(axis, xa):
    if axis not in xa.coords or xa.coords[axis].size < 2:
        return None
    return float(xa.coords[axis][1] - xa.coords[axis][0])

In [7]:
def segment_to_view(i, lif, lif_path, classifier_path):
    img = lif.images[i]
    filename = "".join(os.path.basename(lif_path).lower().replace(".lif", ""))+"__"+("".join(img.path))
    print("Now segmenting the file {}".format(filename))
    try:

        if img.dims == ('T', 'Z', 'Y', 'X'):
            image = img.asarray()
            xa = img.asxarray()

            # liffile coordinates are typically in meters → convert to µm
            x_um = step("X", xa) * 1e6 if step("X", xa) is not None else None
            z_um = step("Z", xa) * 1e6 if step("Z", xa) is not None else None
            # spacing_um = [x_um, y_um, z_um]

            t0 = image[0,:,:,:]
            t2 = image[2,:,:,:]
            gel_matrix = apply_weka_with_exact_preprocessing(t0, classifier_path)
        
            vasculature_segmentation = (gel_matrix == 0).astype(int)
            vasculature_labels = label(vasculature_segmentation)

            table = regionprops_table(vasculature_labels, properties=('label', 'area'),)

            condition = (table['area'] >= 20)
            input_labels = table['label']
            output_labels = input_labels * condition
            output_labels = util.map_array(vasculature_labels, input_labels, output_labels)
            clean_vasculature_segmentation = output_labels > 0
            # clean_gel_segmentation = (clean_vasculature_segmentation == 0).astype(int)
            
            t2 = convert(t2, 0, 255, np.uint8)
            t0 = convert(t0, 0, 255, np.uint8)

            rescaled_t0 = rescale(scale = (z_um/x_um,1,1), image=(t0 ), anti_aliasing = False)
            rescaled_t2 = rescale(scale = (z_um/x_um,1,1), image=(t2 ), anti_aliasing = False)
            rescaled_vasculature_segmentation = rescale(scale = (z_um/x_um,1,1), image=(clean_vasculature_segmentation ), anti_aliasing = False, order=0, preserve_range=True).astype(clean_vasculature_segmentation.dtype)
        else:
            print("Wrong dimensions, try another image")
    except:
        print("Segmentation failed for ", filename)
    IJ.run("Close All")
    return rescaled_t0, rescaled_t2, rescaled_vasculature_segmentation


In [8]:
def select_image():
    root = Tk()
    root.withdraw()
    root.attributes('-topmost', True)
    lif_path_str = filedialog.askopenfilename(parent=root,
    title="Select a .lif file",
    filetypes=[("LIF files",".lif"), ("All files",".*")])
    root.destroy()

    if lif_path_str:
        lif_path = Path(lif_path_str)
        print("Selected:", lif_path)
    else:
        print("No file selected")
    print("Here are all the images in your lif....")
    with LifFile(lif_path) as lif:
        number_of_lifs = len(lif.images)

        for i in range(number_of_lifs):
            print(i, lif_path)
    return(lif_path)

In [10]:
classifier_path="C:/Users/taylorhearn/git_repos/image_quantification/Vasculature/classifier.model" # you might need to update this to wherever you have saved the classifier model
lif_path = select_image()

Selected: Z:\Marina\Stellaris\2025.10.22_Permeab M7\2025.10.22_FL12.lif
Here are all the images in your lif....
0 Z:\Marina\Stellaris\2025.10.22_Permeab M7\2025.10.22_FL12.lif
1 Z:\Marina\Stellaris\2025.10.22_Permeab M7\2025.10.22_FL12.lif
2 Z:\Marina\Stellaris\2025.10.22_Permeab M7\2025.10.22_FL12.lif
3 Z:\Marina\Stellaris\2025.10.22_Permeab M7\2025.10.22_FL12.lif
4 Z:\Marina\Stellaris\2025.10.22_Permeab M7\2025.10.22_FL12.lif
5 Z:\Marina\Stellaris\2025.10.22_Permeab M7\2025.10.22_FL12.lif
6 Z:\Marina\Stellaris\2025.10.22_Permeab M7\2025.10.22_FL12.lif
7 Z:\Marina\Stellaris\2025.10.22_Permeab M7\2025.10.22_FL12.lif
8 Z:\Marina\Stellaris\2025.10.22_Permeab M7\2025.10.22_FL12.lif
9 Z:\Marina\Stellaris\2025.10.22_Permeab M7\2025.10.22_FL12.lif
10 Z:\Marina\Stellaris\2025.10.22_Permeab M7\2025.10.22_FL12.lif
11 Z:\Marina\Stellaris\2025.10.22_Permeab M7\2025.10.22_FL12.lif
12 Z:\Marina\Stellaris\2025.10.22_Permeab M7\2025.10.22_FL12.lif
13 Z:\Marina\Stellaris\2025.10.22_Permeab M7\2025.10.

In [ ]:
i = int(input("Enter the image index to segment (i in the list above): "))
with LifFile(lif_path) as lif:       
    t0, t2, segmentation = segment_to_view(i,lif, lif_path, classifier_path)

Now segmenting the file 2025.10.22_fl12__FL12_LNeg_dev2/P 2


In [ ]:
viewer = napari.Viewer()
viewer.add_image(t0, name="t0")
viewer.add_image(t2, name="t2")
viewer.add_labels(segmentation, name="vasculature_segmentation", colormap={1: np.array(to_rgba("limegreen"), dtype=float)})


RuntimeError: wrapped C/C++ object of type QMenuItemAction has been deleted